In [1]:
import time
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import norm

from model import TransferModel
from config import SlimConfig
from data_processing import DataProcessor
from real_score import load_line, set_line_context, score_line   # your copy with the settings loop

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- settings -------------------------------------------------------------
K, LAM, ALPHA = 20, 100, 2.0  # model config you'd submit (see validate_loco.ipynb)
CONTEXT = "A"                  # which context's controls to test against
N_TARGETS = 30                 # sampled challenge targets (more = slower, steadier numbers)
SEED = 0
FDR = 0.05                     # BH threshold for "significant"

In [2]:
DATASETS = {
    "K562": "../../data/replogle_nadig/K562_gwps_raw_bulk_01.h5ad",
    "RPE1": "../../data/replogle_nadig/rpe1_raw_bulk_01.h5ad",
    "HepG2": "../../data/replogle_nadig/nadig_hepg2_pseudobulk.h5ad",
    "Jurkat": "../../data/replogle_nadig/nadig_jurkat_pseudobulk.h5ad",
}

In [3]:
gene_names = pd.read_csv("../../data/controls/gene_names.csv")["gene_name"].tolist()
dp = DataProcessor(gene_names)
kd = 0.9

In [4]:
train_names = list(DATASETS)                                   # all four training lines

model = TransferModel(gene_names, '../../data/embeddings/gene_string_embeddings.v0.3.h5', SlimConfig())
model.config.K, model.config.lambda_reg = K, LAM

measured_genes = {name: dp.load_dataset(name).var_names for name in train_names}
model.fit(*dp.build_Y(train_names, genes="union"), measured_genes)

  Fitted on 15627 perts, STRING coverage: 15118/15627
  direct transfer from ('K562',): 9675 targets have a measurement


In [5]:
xaira_data = load_line('HCT116', gene_names)
vcc2025_data = load_line('H1', gene_names, target_umi = 20000)

lines_dict = {
    'HCT116': xaira_data,
    'H1': vcc2025_data
}

HCT116: 18400 controls, 198 targets (cells per target median 184, min 100), 100 dropped (7s)
H1: 18400 controls, 97 targets (cells per target median 400, min 139), 3 dropped, control UMI/cell median 54616 -> 20009 (p=0.366) (50s)


In [6]:
ALPHA = 2.0          # match the submission (the notebook's first cell has 1.5)
rows = {}
for line_name, line in lines_dict.items():
    for tc in (200, 50):
        for s in (1.0, 0.5, 0.0):
            model.transfer_c = tc
            set_line_context(model, line, dp, train_names)   # resets b_c
            model.b_c = s * model.b_c
            sc, raw = score_line(model, line, {"gain2.5": {"gain": 2.5}},
                                 kd=kd, alpha=ALPHA, seed=SEED, verbose=False)
            rows[(line_name, tc, s)] = sc.loc["gain2.5"]
            print(line_name, tc, s, round(rows[(line_name, tc, s)]["overall"], 3))

model.transfer_c = 200   # back to the submitted value
grid = pd.DataFrame(rows).T
grid.index.names = ["line", "transfer_c", "b_scale"]
grid.round(3)

HCT116 {'K562': np.float64(0.27), 'RPE1': np.float64(0.3), 'HepG2': np.float64(0.19), 'Jurkat': np.float64(0.24)}
HCT116 200 1.0 0.285
HCT116 {'K562': np.float64(0.27), 'RPE1': np.float64(0.3), 'HepG2': np.float64(0.19), 'Jurkat': np.float64(0.24)}
HCT116 200 0.5 0.27
HCT116 {'K562': np.float64(0.27), 'RPE1': np.float64(0.3), 'HepG2': np.float64(0.19), 'Jurkat': np.float64(0.24)}
HCT116 200 0.0 0.235
HCT116 {'K562': np.float64(0.27), 'RPE1': np.float64(0.3), 'HepG2': np.float64(0.19), 'Jurkat': np.float64(0.24)}
HCT116 50 1.0 0.315
HCT116 {'K562': np.float64(0.27), 'RPE1': np.float64(0.3), 'HepG2': np.float64(0.19), 'Jurkat': np.float64(0.24)}
HCT116 50 0.5 0.346
HCT116 {'K562': np.float64(0.27), 'RPE1': np.float64(0.3), 'HepG2': np.float64(0.19), 'Jurkat': np.float64(0.24)}
HCT116 50 0.0 0.348
H1 {'K562': np.float64(0.28), 'RPE1': np.float64(0.23), 'HepG2': np.float64(0.23), 'Jurkat': np.float64(0.25)}
H1 200 1.0 -0.026
H1 {'K562': np.float64(0.28), 'RPE1': np.float64(0.23), 'HepG2': 

pds    mse    fid  reach    jac   nmae  overall  \
line   transfer_c b_scale                                                      
HCT116 200        1.0      0.246  1.000  0.044  0.256  0.008  0.156    0.285   
                  0.5      0.300  0.752  0.107  0.287  0.013  0.164    0.270   
                  0.0      0.334  0.644  0.012  0.253  0.017  0.149    0.235   
       50         1.0      0.324  1.000  0.032  0.321  0.007  0.204    0.315   
                  0.5      0.449  1.000  0.051  0.356  0.009  0.213    0.346   
                  0.0      0.497  1.000  0.023  0.358  0.011  0.198    0.348   
H1     200        1.0      0.296  0.000 -0.376  0.052 -0.074 -0.057   -0.026   
                  0.5      0.435  0.000 -0.486  0.056 -0.100 -0.043   -0.023   
                  0.0      0.605  0.000 -0.568  0.067 -0.114 -0.048   -0.010   
       50         1.0      0.377  0.000 -0.265  0.053 -0.051 -0.062    0.009   
                  0.5      0.536  0.000 -0.338  0.062 -0.067 -0.047    0.025   
                  0.0      0.700  0.000 -0.398  0.066 -0.078 -0.051    0.040   

                            n_sig  
line   transfer_c b_scale          
HCT116 200        1.0       396.5  
                  0.5       158.0  
                  0.0       123.0  
       50         1.0       582.5  
                  0.5       352.5  
                  0.0       299.5  
H1     200        1.0       691.0  
                  0.5       357.0  
                  0.0       315.0  
       50         1.0      1176.0  
                  0.5       914.0  
                  0.0       816.0

In [7]:
grid.sort_values(by = 'overall', ascending = False)

pds       mse       fid     reach       jac  \
line   transfer_c b_scale                                                     
HCT116 50         0.0      0.497463  1.000000  0.023310  0.358427  0.010507   
                  0.5      0.449330  1.000000  0.050942  0.355919  0.009055   
                  1.0      0.324054  1.000000  0.031530  0.321235  0.007044   
       200        1.0      0.245740  1.000000  0.044468  0.256294  0.008006   
                  0.5      0.299727  0.751824  0.106960  0.287051  0.013201   
                  0.0      0.334201  0.644172  0.012200  0.253487  0.016871   
H1     50         0.0      0.699745  0.000000 -0.397829  0.065600 -0.077895   
                  0.5      0.536286  0.000000 -0.337870  0.061835 -0.066507   
                  1.0      0.376536  0.000000 -0.265353  0.052978 -0.050726   
       200        0.0      0.605379  0.000000 -0.567705  0.066557 -0.113656   
                  0.5      0.434500  0.000000 -0.485585  0.055848 -0.099643   
                  1.0      0.295850  0.000000 -0.375729  0.052460 -0.074331   

                               nmae   overall   n_sig  
line   transfer_c b_scale                              
HCT116 50         0.0      0.197780  0.347915   299.5  
                  0.5      0.213197  0.346407   352.5  
                  1.0      0.203841  0.314617   582.5  
       200        1.0      0.155946  0.285076   396.5  
                  0.5      0.163730  0.270415   158.0  
                  0.0      0.148824  0.234959   123.0  
H1     50         0.0     -0.050779  0.039807   816.0  
                  0.5     -0.046612  0.024522   914.0  
                  1.0     -0.061618  0.008636  1176.0  
       200        0.0     -0.048391 -0.009636   315.0  
                  0.5     -0.043111 -0.022998   357.0  
                  1.0     -0.056693 -0.026407   691.0

In [ ]:
SEED2 = 1
check = {}
for tc, s in [(200, 1.0), (50, 0.5), (50, 0.0)]:
    model.transfer_c = tc
    set_line_context(model, vcc2025_data, dp, train_names)
    model.b_c = s * model.b_c
    sc, _ = score_line(model, vcc2025_data, {"gain2.5": {"gain": 2.5}},
                       kd=kd, alpha=ALPHA, seed=SEED2, verbose=False)
    check[(tc, s)] = sc.loc["gain2.5"]
model.transfer_c = 200
pd.DataFrame(check).T.round(3)

H1 {'K562': np.float64(0.28), 'RPE1': np.float64(0.23), 'HepG2': np.float64(0.23), 'Jurkat': np.float64(0.25)}
H1 {'K562': np.float64(0.28), 'RPE1': np.float64(0.23), 'HepG2': np.float64(0.23), 'Jurkat': np.float64(0.25)}
H1 {'K562': np.float64(0.28), 'RPE1': np.float64(0.23), 'HepG2': np.float64(0.23), 'Jurkat': np.float64(0.25)}


pds  mse    fid  reach    jac   nmae  overall  n_sig
200 1.0  0.295  0.0 -0.375  0.044 -0.076 -0.052   -0.027  695.0
50  0.5  0.532  0.0 -0.339  0.078 -0.071 -0.041    0.026  917.0
    0.0  0.705  0.0 -0.389  0.083 -0.081 -0.044    0.046  816.0

In [9]:
pd.DataFrame(check).T.round(3).sort_values(by = 'overall', ascending = False)

pds  mse    fid  reach    jac   nmae  overall  n_sig
50  0.0  0.705  0.0 -0.389  0.083 -0.081 -0.044    0.046  816.0
    0.5  0.532  0.0 -0.339  0.078 -0.071 -0.041    0.026  917.0
200 1.0  0.295  0.0 -0.375  0.044 -0.076 -0.052   -0.027  695.0

In [ ]:
GRID = {}

for N in (300,7000, 1500):
    for base_gain in (1, 2.5):
        GRID[f'{N}_2.5_{base_gain}'] = {
            "gain": base_gain,
            "n_call": N,
            "gain_top": 2.5
        }
GRID[f'{N}_2.5_2.5'] = {
    "gain": 2.5,
    "n_call": N,
    "gain_top": 2.5
}

GRID

In [ ]:
results = {}

for name, line in lines_dict.items():
    set_line_context(model, line, dp, train_names)
    scaled_score, raw_score = score_line(model, line, GRID, kd=kd, alpha=ALPHA, seed=SEED)
    results[name] = [scaled_score, raw_score]

results

In [ ]:
results['H1'][1]

In [ ]:
merged = pd.concat(
    {(line, kind): df
     for line, (scaled, raw) in results.items()
     for kind, df in (("scaled", scaled), ("raw", raw))},
    names=["line", "kind", "setting"],
)
merged.reset_index(inplace = True)

In [ ]:
merged.sort_values('overall', ascending = False)

In [ ]:
merged_no_lines = merged.copy().drop(columns = 'line')

In [ ]:
merged_no_lines.groupby(['kind', 'setting']).mean().reset_index().sort_values(by = 'overall', ascending = False)

In [ ]:
merged[merged['kind'] == 'scaled']